# Ders 03 - Ajan Tasarım Desenleri

Bu derste, etkili yapay zeka ajanları oluşturmak için üç temel tasarım desenini inceliyoruz:

1. **Net Ajan Talimatları** — Ajan davranışını yönlendiren kesin, rol tanımlayan istemler oluşturma  
2. **Pydantic Modelleri ile Yapılandırılmış Çıktı** — Ajanların öngörülebilir, doğrulanmış veriler döndürmesini sağlama  
3. **Tek Sorumluluklu Ajanlar** — Her biri bir işi iyi yapan odaklanmış ajanlar tasarlama  

Her deseni, destinasyon öneren bir **seyahat destinasyonu önericisi** senaryosunda uygulayacağız; böylece destinasyonlar öneren, uygunluğu kontrol eden ve lojistiği yöneten bir sistem adım adım inşa edeceğiz.


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Model 1: Net Ajan Talimatları

En etkili model aynı zamanda en basit olanıdır: ajanın için net, ayrıntılı talimatlar yazmak.

İyi talimatlar şunları tanımlar:
- **Kimin** ajan olduğu (kişilik ve ton)
- **Ne** yapması gerektiği (adım adım sorumluluklar)
- **Nasıl** davranması gerektiği (kısıtlamalar ve tarz)

Aşağıda, ürettiği her yanıtı şekillendiren açık talimatlara sahip bir seyahat konsiyerj ajanı oluşturuyoruz.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Desen 2: Pydantic Modelleri ile Yapılandırılmış Çıktı

Serbest biçimli metin sohbet için faydalıdır, ancak sonraki sistemler yapılandırılmış verilere ihtiyaç duyar.
**Pydantic modelleri**ni bir **araç fonksiyonu** ile eşleştirerek şunları yapabiliriz:

- Ajanın çıktısı için kesin bir şema tanımlamak
- Yanıtları otomatik olarak doğrulamak
- Ajan sonuçlarını uygulama mantığına güvenilir şekilde entegre etmek

Ayrıca, ajanın önerilerini gerçek verilere dayandırması için varış yeri detaylarını döndüren bir araç da tanıtıyoruz.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Model 3: Tek Sorumluluklu Ajanlar

Karmaşık görevler, her biri tek bir sorumluluğa sahip birden çok odaklanmış ajan arasında işin bölünmesinden fayda sağlar:

- Yerler ve müsaitlik hakkında bilgi sahibi bir **Varış Yeri Uzmanı**
- Uçuşlar, oteller ve yolculuk programlarıyla ilgilenen bir **Lojistik Planlayıcı**

Bu, yazılım mühendisliğinde *kaygıların ayrılması* prensibini yansıtır — her ajan bağımsız olarak test edilmesi, bakımı ve geliştirilmesi daha kolaydır.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Özet

Bu derste, seyahat öneri senaryosuna üç ajan tasarım deseni uyguladık:

| Desen | Temel Fikir | Fayda |
|---|---|---|
| **Net Talimatlar** | Persona, sorumluluklar ve kısıtlamaları baştan tanımla | Tutarlı, marka uyumlu ajan davranışı |
| **Yapılandırılmış Çıktı** | Yanıt formatı olarak Pydantic modellerini kullan | Doğrulanmış, makine tarafından okunabilir sonuçlar |
| **Tek Sorumluluk** | Her ajana odaklanmış tek bir iş ver | Test edilmesi, bakımı ve birleştirmesi daha kolay |

Bu desenler doğal olarak bir araya gelir — net talimatları yapılandırılmış çıktı ile tek sorumluluklu ajan içinde birleştirerek sağlam, üretime hazır sistemler oluşturabilirsiniz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
